# Visual inspection of missed human-added boxes

Baselines recall ~0.51 of the boxes the annotator added by hand, against
~0.97 of the boxes carried over from teacher pre-annotation. This notebook
renders crops around a sample of the missed boxes so the failure mode can be
characterised: genuinely difficult bolls (shadow, occlusion, partial view)
versus an artefact of how those boxes were drawn.

Needs in Drive: `gold_predictions.json`, `gold_v2_verified.zip`,
`gold_v2_coco_c30.json`, `gold_v2_tiles.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
import os, json, glob, random
import numpy as np, pandas as pd
from collections import defaultdict
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2

preds_all = json.load(open(f'{DRIVE}/gold_predictions.json'))
!rm -rf /content/gold_ann /content/gold
!unzip -q {DRIVE}/gold_v2_verified.zip -d /content/gold_ann
!unzip -q {DRIVE}/gold_v2_tiles.zip -d /content/gold
ann = glob.glob('/content/gold_ann/**/instances_default.json', recursive=True)[0]
verified = json.load(open(ann))
teacher  = json.load(open(f'{DRIVE}/gold_v2_coco_c30.json'))
TILES = '/content/gold/gold_v2'
print('ready')

In [ ]:
def to_xyxy(d):
    names = {im['id']: im['file_name'].split('/')[-1] for im in d['images']}
    out = defaultdict(list)
    for a in d['annotations']:
        x, y, w, h = a['bbox']
        out[names[a['image_id']]].append([x, y, x+w, y+h])
    for n in names.values(): out.setdefault(n, [])
    return dict(out)

def iou_matrix(A, B):
    if len(A) == 0 or len(B) == 0: return np.zeros((len(A), len(B)), np.float32)
    A = np.asarray(A, np.float32); B = np.asarray(B, np.float32)
    ix1 = np.maximum(A[:,0,None], B[None,:,0]); iy1 = np.maximum(A[:,1,None], B[None,:,1])
    ix2 = np.minimum(A[:,2,None], B[None,:,2]); iy2 = np.minimum(A[:,3,None], B[None,:,3])
    inter = np.clip(ix2-ix1,0,None)*np.clip(iy2-iy1,0,None)
    aA = ((A[:,2]-A[:,0])*(A[:,3]-A[:,1]))[:,None]
    aB = ((B[:,2]-B[:,0])*(B[:,3]-B[:,1]))[None,:]
    u = aA + aB - inter
    return np.divide(inter, u, out=np.zeros_like(inter), where=u>0)

gt_all, gt_teach = to_xyxy(verified), to_xyxy(teacher)

# human-added boxes, and whether the model recovers each at its operating point
MODEL, THR = 'yolo11s', 0.35
preds = preds_all[MODEL]

added, hit, miss = [], [], []
for t, boxes in gt_all.items():
    if not boxes: continue
    tb = gt_teach.get(t, [])
    M = iou_matrix(boxes, tb)
    is_new = (M.max(1) < 0.5) if M.shape[1] else np.ones(len(boxes), bool)
    new_boxes = [b for b, n in zip(boxes, is_new) if n]
    if not new_boxes: continue
    p = [b[:4] for b in preds.get(t, []) if b[4] >= THR]
    P = iou_matrix(new_boxes, p)
    rec = (P.max(1) >= 0.5) if P.shape[1] else np.zeros(len(new_boxes), bool)
    for b, r in zip(new_boxes, rec):
        (hit if r else miss).append((t, b))
        added.append((t, b, bool(r)))
print(f'human-added: {len(added):,} | recovered {len(hit):,} | missed {len(miss):,} '
      f'({len(hit)/len(added)*100:.1f}% recall)')

## Are the missed boxes different in size or brightness?

Before looking at crops, two quantitative checks. If missed boxes are
systematically smaller or darker than recovered ones, that characterises the
failure mode without any subjective judgement.

In [ ]:
def stats(group, label):
    sizes, brights = [], []
    cache = {}
    for t, b in group[:4000]:
        if t not in cache:
            img = cv2.imread(f'{TILES}/{t}')
            cache[t] = img
        img = cache[t]
        if img is None: continue
        x1, y1, x2, y2 = [int(max(0, v)) for v in b]
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: continue
        sizes.append(max(x2-x1, y2-y1))
        brights.append(float(crop.mean()))
    return dict(subset=label, n=len(sizes),
                med_size_px=round(float(np.median(sizes)),1),
                med_brightness=round(float(np.median(brights)),1))

rows = [stats(miss, 'missed'), stats(hit, 'recovered')]
# reference: teacher-derived boxes
kept = []
for t, boxes in gt_all.items():
    tb = gt_teach.get(t, [])
    if not boxes or not tb: continue
    M = iou_matrix(boxes, tb)
    for b, m in zip(boxes, M.max(1) >= 0.5):
        if m: kept.append((t, b))
random.seed(0); random.shuffle(kept)
rows.append(stats(kept, 'teacher-derived'))
print(pd.DataFrame(rows).to_string(index=False))

## Crops

Each panel is a 96 px window centred on one missed human-added box. The red
box is the annotation; any model detections above threshold in the window are
drawn in yellow.

In [ ]:
random.seed(7)
sample = random.sample(miss, 24)
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
PAD = 48
for ax, (t, b) in zip(axes.ravel(), sample):
    img = cv2.cvtColor(cv2.imread(f'{TILES}/{t}'), cv2.COLOR_BGR2RGB)
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    x1, y1 = int(max(0, cx-PAD)), int(max(0, cy-PAD))
    x2, y2 = int(min(640, cx+PAD)), int(min(640, cy+PAD))
    crop = img[y1:y2, x1:x2].copy()
    cv2.rectangle(crop, (int(b[0]-x1), int(b[1]-y1)),
                  (int(b[2]-x1), int(b[3]-y1)), (255,0,0), 1)
    for pb in preds.get(t, []):
        if pb[4] < THR: continue
        if x1 <= (pb[0]+pb[2])/2 <= x2 and y1 <= (pb[1]+pb[3])/2 <= y2:
            cv2.rectangle(crop, (int(pb[0]-x1), int(pb[1]-y1)),
                          (int(pb[2]-x1), int(pb[3]-y1)), (255,220,0), 1)
    ax.imshow(crop); ax.axis('off')
    ax.set_title(f'{int(max(b[2]-b[0], b[3]-b[1]))}px', fontsize=8)
fig.suptitle(f'Human-added boxes missed by {MODEL} at conf>={THR} '
             '(red: annotation, yellow: model)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{DRIVE}/missed_added_boxes.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Same sample, recovered boxes, for comparison
sample = random.sample(hit, 24)
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
for ax, (t, b) in zip(axes.ravel(), sample):
    img = cv2.cvtColor(cv2.imread(f'{TILES}/{t}'), cv2.COLOR_BGR2RGB)
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    x1, y1 = int(max(0, cx-PAD)), int(max(0, cy-PAD))
    x2, y2 = int(min(640, cx+PAD)), int(min(640, cy+PAD))
    crop = img[y1:y2, x1:x2].copy()
    cv2.rectangle(crop, (int(b[0]-x1), int(b[1]-y1)),
                  (int(b[2]-x1), int(b[3]-y1)), (0,200,0), 1)
    ax.imshow(crop); ax.axis('off')
    ax.set_title(f'{int(max(b[2]-b[0], b[3]-b[1]))}px', fontsize=8)
fig.suptitle('Human-added boxes the model does recover (green)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{DRIVE}/recovered_added_boxes.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
import os, cv2
print('TILES dir exists:', os.path.exists(TILES))
print('files there:', len(os.listdir(TILES)) if os.path.exists(TILES) else 0)
print('sample listing:', os.listdir(TILES)[:2] if os.path.exists(TILES) else None)

t, b = miss[0]
print('tile name from annotations:', repr(t))
print('constructed path:', f'{TILES}/{t}')
print('exists:', os.path.exists(f'{TILES}/{t}'))
img = cv2.imread(f'{TILES}/{t}')
print('imread result:', None if img is None else img.shape)

In [ ]:
for f in ['missed_added_boxes.png', 'recovered_added_boxes.png']:
    p = f'{DRIVE}/{f}'
    print(f, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else '')

In [ ]:
from IPython.display import Image, display
display(Image(f'{DRIVE}/missed_added_boxes.png'))
display(Image(f'{DRIVE}/recovered_added_boxes.png'))